# 11 · Is BV domain clustering a shared-scene effect? (subject split + VEDB)

**Question.** BabyView crops carry their surroundings. If categories from one CDI domain tend to appear in the same
scenes (food on the same high-chair tray, furniture in the same living room), within-domain distances could shrink
because of shared context rather than shared object appearance. THINGS photos come from unrelated sources.

**Two tests.**
1. **Subject split.** Split BV subjects (one child per subject ID; we have no household metadata, and
   subject IDs that share a prefix are not siblings) into halves A/B. Category means from half A are compared
   with means from half B, so the two categories in a pair never share a child, home, room, session or lighting. Compare cluster strength in this
   **cross-half RDM** with the **within-half RDM** (A vs A, where pairs can share scenes). VEDB is split by
   session; THINGS by image (half of each category's images, since images are independent).
2. **VEDB.** Adult head-camera video shares BV's egocentric, real-room context but not an infant's experience.
   If VEDB clusters as much as BV, the BV-vs-THINGS gap is a property of naturalistic egocentric input.

**Decisions**
- Same exemplars as the manuscript: 2025.2 packed BV shards and VEDB shards, CLIP image–label similarity ≥ 0.27,
  crops L2-normalized, BV subject 00270001 excluded; THINGS = all images per category. A check cell
  confirms full-data means reproduce manuscript Stage 0 (2025.2 BV, THINGS).
- Each half's category means are **z-scored across categories separately** (manuscript convention). The
  cross-half RDM is symmetrized, (A_i·B_j + A_j·B_i)/2; the diagonal is unused.
- `K_SPLITS` random splits; a category is kept only if it has ≥ `MIN_N` exemplars in **both halves of every split**.
- Three measures of cluster strength (see `09_cluster_strength` for why raw Δ alone misleads):
  raw Δ = mean between-domain − mean within-domain distance; Δ / SD of that RDM's distances; and
  **AUC = P(between-domain distance > within-domain distance)**, which is scale-free.
- Dataset differences: **within-domain category bootstrap** on the split-averaged cross-half RDM (self-pairs
  dropped), two-sided. Not a label shuffle (it removes structure from both datasets).
- What this cannot remove: context that recurs in *every* home (food is usually on a table). The subject split
  removes *shared scenes*, not *typical settings*; VEDB speaks to the latter.

**Outputs:** `outputs/context_split/results/` and `figures/`. Per-video caches go to `outputs/context_split/cache/`
(git-ignored, ~0.8 GB; rebuilt in ~25 min when missing or `RECOMPUTE_CACHE = True`).

In [ ]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

# Manuscript figure style (same as analysis/qwen-open-vocab/fig_style.py): Lato, bold axis labels and legends.
import matplotlib.axes, matplotlib.axis, matplotlib.figure
from matplotlib import font_manager
for _f in (Path.cwd().resolve().parents[1] / "talk-figures" / "unr-2026" / "fonts").glob("*.ttf"):
    font_manager.fontManager.addfont(str(_f))
plt.rcParams.update({
    "font.family": ["Lato", "DejaVu Sans", "sans-serif"], "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.labelweight": "bold", "figure.titlesize": 12, "figure.titleweight": "bold",
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.labelcolor": "#222222",
    "text.color": "#222222", "xtick.color": "#444444", "ytick.color": "#444444"})
def _bold_legend(orig):
    if getattr(orig, "_bold", False):
        return orig
    def legend(self, *args, **kw):
        prop = dict(kw.pop("prop", {}) or {}); prop.setdefault("weight", "bold")
        prop.setdefault("size", kw.pop("fontsize", plt.rcParams["legend.fontsize"]))
        return orig(self, *args, prop=prop, **kw)
    legend._bold = True
    return legend
matplotlib.axes.Axes.legend = _bold_legend(matplotlib.axes.Axes.legend)
matplotlib.figure.Figure.legend = _bold_legend(matplotlib.figure.Figure.legend)
_tick_init = matplotlib.axis.Tick.__init__
if not getattr(_tick_init, "_bold", False):
    def _bold_tick(self, *a, **k):                       # bold tick labels (no rcParam for tick label weight)
        _tick_init(self, *a, **k); self.label1.set_fontweight("bold"); self.label2.set_fontweight("bold")
    _bold_tick._bold = True; matplotlib.axis.Tick.__init__ = _bold_tick
    _grid = matplotlib.axes.Axes.grid                    # no grid lines in any figure
    matplotlib.axes.Axes.grid = lambda self, visible=None, which="major", axis="both", **kw: _grid(self, False, which=which, axis=axis)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

# Inputs for this notebook (read only).
MANUSCRIPT_SCRIPTS = MANUSCRIPT_DIR / "scripts"
BV_PACKED = Path("/ccn2a/dataset/babyview/2025.2/outputs/image_embeddings/babyview_full")
VEDB_PACKED = Path("/ccn2a/dataset/babyview/related_data/vedb_1612/outputs/image_embeddings/vedb_full_adults")
THINGS_CLIP_NPY = Path("/ccn2/dataset/babyview/outputs_20250312/things_image_embeddings/clip_embeddings")
THINGS_DINOV3_NPY = Path("/ccn2/dataset/babyview/outputs_20250312/image_embeddings/"
                         "things_bv_overlapping_categories_corrected/facebook_dinov3-vitb16-pretrain-lvd1689m")
STAGE0_BV_2025_2 = MANUSCRIPT_DIR / "exemplar_set_embeddings" / "2025.2" / "valid129"
STAGE0_THINGS = MANUSCRIPT_DIR / "exemplar_set_embeddings" / "valid129"

CTX_OUT = OUT / "context_split"
CTX_CACHE = CTX_OUT / "cache"
CTX_RESULTS = CTX_OUT / "results"
CTX_FIGURES = CTX_OUT / "figures"
for d in (CTX_CACHE, CTX_RESULTS, CTX_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
K_SPLITS = 50           # random half splits
MIN_N = 5               # exemplars per category required in each half of every split
MIN_SIM = 0.27          # CLIP image–label threshold (manuscript)
N_BOOT = 2000           # category bootstrap for raw / scaled Δ differences
N_BOOT_AUC = 1000       # category bootstrap for AUC (slower: ranks per draw)
RECOMPUTE_CACHE = False
EXCLUDE_BV_SUBJECTS = ("00270001",)

LABEL = {"bv": "BabyView", "vedb": "VEDB", "things": "THINGS"}
COLOR = {"bv": "#0D47A1", "vedb": "#E07B39", "things": "#90CAF9"}   # same as notebooks 07-09
COMPARISONS = {"bv_things": ["bv", "things"], "bv_vedb_things": ["bv", "vedb", "things"]}

## Shared helpers

In [ ]:
def load_cdi_semantic_map() -> dict[str, str]:
    """``uni_lemma`` → CDI semantic domain from ``data/cdi_words.csv`` (same as notebook 09)."""
    df = pd.read_csv(DATA_DIR / "cdi_words.csv", usecols=["uni_lemma", "category"])
    df["uni_lemma"] = df["uni_lemma"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=["uni_lemma"], keep="first").set_index("uni_lemma")["category"].to_dict()


CATS = sorted(l.strip().lower() for l in VALID129_TXT.read_text().splitlines() if l.strip())
CDI = load_cdi_semantic_map()
print(len(CATS), "categories in valid129")

## Stage A · per-video caches

One pass over each packed store: for every (video, category), the sum and count of L2-normalized crop vectors.
Same row filter as the manuscript's `packed_crop_embeddings.category_means`. THINGS: every per-image vector.

In [ ]:
import sys
sys.path.insert(0, str(MANUSCRIPT_SCRIPTS))
from packed_crop_embeddings import (_row_mask, _vectors_for_mask, iter_shard_paths, l2_normalize_rows,
                                    subject_id_from_video_id)


def build_video_cache(emb_root: Path, out: Path, exclude_subjects=()) -> None:
    cix = {c: i for i, c in enumerate(CATS)}
    rec = {m: {"vid": [], "cat": [], "n": [], "sum": []} for m in MODELS}
    vids = []
    paths = list(iter_shard_paths(emb_root))
    for k, p in enumerate(paths):
        sid = subject_id_from_video_id(p.stem)
        if sid is not None and sid in set(exclude_subjects):
            continue
        with np.load(p, allow_pickle=True) as z:
            sh = {kk: z[kk] for kk in z.files}
        vi = len(vids); vids.append(p.stem)
        names = np.array([str(x).strip().lower() for x in sh["class_name"]])
        for m in MODELS:
            mask = _row_mask(sh, categories=set(CATS), min_sim=MIN_SIM, passed_only=False, model=m)
            if not mask.any():
                continue
            v = l2_normalize_rows(_vectors_for_mask(sh, mask, m)); nm = names[mask]
            for c in np.unique(nm):
                sel = nm == c
                r = rec[m]; r["vid"].append(vi); r["cat"].append(cix[c]); r["n"].append(int(sel.sum()))
                r["sum"].append(v[sel].sum(0).astype(np.float32))
        if k % 500 == 0:
            print(f"  {emb_root.name}: {k}/{len(paths)}", flush=True)
    np.savez(out, cats=np.array(CATS), vids=np.array(vids),
             **{f"{m}_{f}": np.asarray(rec[m][f]) for m in rec for f in ("vid", "cat", "n")},
             **{f"{m}_sum": np.stack(rec[m]["sum"]) for m in rec})


def build_things_cache(out: Path) -> None:
    def subdir(root, c):
        d = root / c
        if d.is_dir():
            return d
        hit = [p for p in root.iterdir() if p.is_dir() and p.name.lower() == c]
        return hit[0] if hit else None
    rows = {m: [] for m in MODELS}; meta = {m: [] for m in MODELS}
    for c in CATS:
        for m, root in (("clip", THINGS_CLIP_NPY), ("dinov3", THINGS_DINOV3_NPY)):
            d = subdir(root, c)
            for f in (sorted(d.rglob("*.npy")) if d is not None else []):
                v = np.load(f).astype(np.float64).ravel(); v /= max(np.linalg.norm(v), 1e-12)
                rows[m].append(v.astype(np.float32)); meta[m].append((c, f.stem))
    np.savez(out, **{f"{m}_X": np.stack(rows[m]) for m in MODELS},
             **{f"{m}_cat": np.array([a for a, _ in meta[m]]) for m in MODELS},
             **{f"{m}_stem": np.array([b for _, b in meta[m]]) for m in MODELS})


CACHE = {"bv": CTX_CACHE / "bv_2025_2_video_category_sums.npz",
         "vedb": CTX_CACHE / "vedb_video_category_sums.npz",
         "things": CTX_CACHE / "things_image_vectors.npz"}
if RECOMPUTE_CACHE or not CACHE["bv"].exists():
    build_video_cache(BV_PACKED, CACHE["bv"], EXCLUDE_BV_SUBJECTS)
if RECOMPUTE_CACHE or not CACHE["vedb"].exists():
    build_video_cache(VEDB_PACKED, CACHE["vedb"])          # VEDB sessions are not BV subject IDs: no exclusion
if RECOMPUTE_CACHE or not CACHE["things"].exists():
    build_things_cache(CACHE["things"])
Z = {ds: np.load(p) for ds, p in CACHE.items()}
assert list(Z["bv"]["cats"]) == CATS and list(Z["vedb"]["cats"]) == CATS
print({ds: Path(p).stat().st_size // 2**20 for ds, p in CACHE.items()}, "MB")

## Exemplar tables and split helpers

In [ ]:
def unit_of(ds: str, vid: str) -> str:
    """Split unit. BV: subject ID (one child per ID). VEDB: session."""
    return vid.split("_")[0] if ds == "bv" else vid


def exemplars(ds: str, model: str) -> dict:
    """Rows = (unit, category) sums for BV/VEDB, single images for THINGS."""
    z = Z[ds]
    if ds == "things":
        return dict(unit=z[f"{model}_stem"], cat=np.array([CATS.index(c) for c in z[f"{model}_cat"]]),
                    n=np.ones(len(z[f"{model}_cat"]), int), X=z[f"{model}_X"].astype(np.float64))
    units = np.array([unit_of(ds, v) for v in z["vids"]])
    return dict(unit=units[z[f"{model}_vid"]], cat=z[f"{model}_cat"], n=z[f"{model}_n"],
                X=z[f"{model}_sum"].astype(np.float64))


def category_means(d: dict, sel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    X = np.zeros((len(CATS), d["X"].shape[1])); n = np.zeros(len(CATS))
    np.add.at(X, d["cat"][sel], d["X"][sel]); np.add.at(n, d["cat"][sel], d["n"][sel])
    return X / np.maximum(n, 1)[:, None], n


def split_mask(ds: str, d: dict, k: int) -> np.ndarray:
    """Half A of split k: random half of units (BV, VEDB) or of each category's images (THINGS)."""
    rng = np.random.default_rng(1000 + k)
    if ds == "things":
        sel = np.zeros(len(d["cat"]), bool)
        for c in np.unique(d["cat"]):
            ix = rng.permutation(np.flatnonzero(d["cat"] == c)); sel[ix[: len(ix) // 2]] = True
        return sel
    units = np.unique(d["unit"]); in_a = set(rng.permutation(units)[: len(units) // 2])
    return np.array([u in in_a for u in d["unit"]])


def eligible(ds: str, d: dict) -> np.ndarray:
    ok = np.ones(len(CATS), bool)
    for k in range(K_SPLITS):
        sel = split_mask(ds, d, k); _, na = category_means(d, sel); _, nb = category_means(d, ~sel)
        ok &= (na >= MIN_N) & (nb >= MIN_N)
    return ok


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def cosine_distance(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A = A / np.linalg.norm(A, axis=1, keepdims=True); B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return 1 - A @ B.T


def deltas(R: np.ndarray, sem) -> dict:
    """Global and per-domain Δ = mean between − mean within (upper triangle), plus the RDM's SD."""
    sem = np.asarray(sem); iu = np.triu_indices(len(sem), 1); v = R[iu]; same = sem[iu[0]] == sem[iu[1]]
    out = {"global": v[~same].mean() - v[same].mean(), "sd": v.std()}
    for dom in sorted(set(sem)):
        a = sem == dom
        if a.sum() >= 2:
            out[dom] = R[np.ix_(a, ~a)].mean() - R[np.ix_(a, a)][np.triu_indices(a.sum(), 1)].mean()
    return out

### Check: full-data means reproduce manuscript Stage 0

Max absolute difference from the Stage 0 raw tables (2025.2 BV, THINGS) should be ~1e-8 (float32 sums).

In [ ]:
for m in MODELS:
    for ds, f in (("bv", STAGE0_BV_2025_2 / f"bv_{m}_exemplar_avg_raw_within_valid129.csv"),
                  ("things", STAGE0_THINGS / f"things_{m}_exemplar_avg_raw_within_valid129.csv")):
        ref = pd.read_csv(f); ref["category"] = ref["category"].str.strip().str.lower()
        ref = ref.set_index("category").loc[CATS].filter(like="dim_").to_numpy()
        d = exemplars(ds, m); full, _ = category_means(d, np.ones(len(d["cat"]), bool))
        err = np.abs(full - ref).max(); print(f"{m:6s} {ds:6s} max |Δ| = {err:.1e}"); assert err < 1e-6

## Run splits (~6 min)

In [ ]:
def run_splits(ds: str, d: dict, cat_idx: np.ndarray, sem: list[str]):
    """Per split: Δ in the within-half (A·A) and cross-half (A·B) RDMs; returns the split-averaged cross RDM."""
    rows, rx_sum = [], 0
    for k in range(K_SPLITS):
        sel = split_mask(ds, d, k)
        a, _ = category_means(d, sel); b, _ = category_means(d, ~sel)
        a, b = zscore_rows(a[cat_idx]), zscore_rows(b[cat_idx])
        rw = cosine_distance(a, a); rx = cosine_distance(a, b); rx = (rx + rx.T) / 2
        dw, dx = deltas(rw, sem), deltas(rx, sem)
        rows.append({"split": k, **{f"within_{q}": v for q, v in dw.items()}, **{f"cross_{q}": v for q, v in dx.items()}})
        rx_sum = rx_sum + rx
    return pd.DataFrame(rows), rx_sum / K_SPLITS


SPLITS, CROSS_RDM, SETS = {}, {}, {}
for m in MODELS:
    D = {ds: exemplars(ds, m) for ds in ("bv", "vedb", "things")}
    ok = {ds: eligible(ds, D[ds]) for ds in D}
    for name, dss in COMPARISONS.items():
        cat_idx = np.flatnonzero(np.logical_and.reduce([ok[ds] for ds in dss]))
        cats = [CATS[i] for i in cat_idx]; sem = [CDI.get(c, "other") for c in cats]
        SETS[(m, name)] = (cats, sem)
        for ds in dss:
            SPLITS[(m, name, ds)], CROSS_RDM[(m, name, ds)] = run_splits(ds, D[ds], cat_idx, sem)
        print(f"{m:6s} {name:15s} {len(cats)} categories; units:",
              {ds: len(np.unique(D[ds]['unit'])) for ds in dss})

## Scale-free AUC and category-bootstrap tests on the split-averaged cross-half RDM

In [ ]:
def auc(R, sem, idx=None):
    """P(between-domain distance > within-domain distance) over category pairs (self-pairs dropped)."""
    sem = np.asarray(sem); idx = np.arange(len(sem)) if idx is None else idx
    s = sem[idx]; r = R[np.ix_(idx, idx)]; iu = np.triu_indices(len(idx), 1); keep = idx[iu[0]] != idx[iu[1]]
    v = r[iu][keep]; same = (s[iu[0]] == s[iu[1]])[keep]; w, b = v[same], v[~same]
    ranks = rankdata(np.concatenate([w, b]))
    return (ranks[len(w):].sum() - len(b) * (len(b) + 1) / 2) / (len(w) * len(b))


def delta_raw_scaled(R, sem, idx):
    s = np.asarray(sem)[idx]; r = R[np.ix_(idx, idx)]; iu = np.triu_indices(len(idx), 1)
    keep = idx[iu[0]] != idx[iu[1]]; v = r[iu][keep]; same = (s[iu[0]] == s[iu[1]])[keep]
    dl = v[~same].mean() - v[same].mean(); return dl, dl / v.std()


def domain_bootstrap(sem, n, seed):
    rng = np.random.default_rng(seed); sem = np.asarray(sem); doms = sorted(set(sem))
    for _ in range(n):
        yield np.concatenate([rng.choice(np.flatnonzero(sem == dd), (sem == dd).sum()) for dd in doms])


def ci_p(x):
    lo, hi = np.percentile(x, [2.5, 97.5]); return lo, hi, float(2 * min((x <= 0).mean(), (x >= 0).mean()))


summary_rows, pair_rows, domain_rows = [], [], []
for (m, name), (cats, sem) in SETS.items():
    dss = COMPARISONS[name]; R = {ds: CROSS_RDM[(m, name, ds)] for ds in dss}
    boot_d = np.array([[delta_raw_scaled(R[ds], sem, idx) for ds in dss] for idx in domain_bootstrap(sem, N_BOOT, 7)])
    boot_a = np.array([[auc(R[ds], sem, idx) for ds in dss] for idx in domain_bootstrap(sem, N_BOOT_AUC, 11)])
    for i, ds in enumerate(dss):
        sp = SPLITS[(m, name, ds)]
        a_lo, a_hi, _ = ci_p(boot_a[:, i] - 0.5)
        summary_rows.append({
            "model": m, "comparison": name, "n_categories": len(cats), "dataset": ds,
            "delta_within_half": sp.within_global.mean(), "delta_cross_half": sp.cross_global.mean(),
            "cross_over_within": (sp.cross_global / sp.within_global).mean(),
            "delta_sd_within_half": (sp.within_global / sp.within_sd).mean(),
            "delta_sd_cross_half": (sp.cross_global / sp.cross_sd).mean(),
            "auc_cross_half": auc(R[ds], sem), "auc_ci_lo": a_lo + 0.5, "auc_ci_hi": a_hi + 0.5})
        for dom in sorted(set(sem)):
            domain_rows.append({"model": m, "comparison": name, "dataset": ds, "cdi_semantic": dom,
                                "n_categories": sem.count(dom),
                                "delta_within_half": sp[f"within_{dom}"].mean() if f"within_{dom}" in sp else np.nan,
                                "delta_cross_half": sp[f"cross_{dom}"].mean() if f"cross_{dom}" in sp else np.nan})
    for i, j in combinations(range(len(dss)), 2):
        a, b = dss[i], dss[j]
        obs = {"delta": SPLITS[(m, name, a)].cross_global.mean() - SPLITS[(m, name, b)].cross_global.mean(),
               "delta_sd": (SPLITS[(m, name, a)].cross_global / SPLITS[(m, name, a)].cross_sd).mean()
                           - (SPLITS[(m, name, b)].cross_global / SPLITS[(m, name, b)].cross_sd).mean(),
               "auc": auc(R[a], sem) - auc(R[b], sem)}
        draws = {"delta": boot_d[:, i, 0] - boot_d[:, j, 0], "delta_sd": boot_d[:, i, 1] - boot_d[:, j, 1],
                 "auc": boot_a[:, i] - boot_a[:, j]}
        for metric in ("delta", "delta_sd", "auc"):
            lo, hi, p = ci_p(draws[metric])
            pair_rows.append({"model": m, "comparison": name, "pair": f"{a}_minus_{b}", "metric": metric,
                              "diff": obs[metric], "ci_lo": lo, "ci_hi": hi, "p_boot_two_sided": p,
                              "frac_splits_positive": float((SPLITS[(m, name, a)].cross_global
                                                             > SPLITS[(m, name, b)].cross_global).mean())
                                                      if metric == "delta" else np.nan})

summary = pd.DataFrame(summary_rows); pairs = pd.DataFrame(pair_rows); per_domain = pd.DataFrame(domain_rows)
summary.to_csv(CTX_RESULTS / "context_split_summary.csv", index=False)
pairs.to_csv(CTX_RESULTS / "context_split_dataset_differences.csv", index=False)
per_domain.to_csv(CTX_RESULTS / "context_split_per_domain.csv", index=False)
for (m, name, ds), sp in SPLITS.items():
    sp.to_csv(CTX_RESULTS / f"context_split_per_split_{m}_{name}_{ds}.csv", index=False)
np.savez(CTX_CACHE / "cross_half_rdms.npz",
         **{f"{m}__{name}__{ds}": R for (m, name, ds), R in CROSS_RDM.items()},
         **{f"{m}__{name}__cats": np.array(SETS[(m, name)][0]) for (m, name) in SETS})
print("wrote", *sorted(p.name for p in CTX_RESULTS.glob("context_split_*.csv"))[:3], "...")

## Results

**Within vs cross half.** `cross_over_within` near 1 means shared scenes contribute little to clustering.

In [ ]:
display(summary.round(3))

**Dataset differences** (cross-half RDM; category bootstrap; `delta` raw, `delta_sd` scaled by RDM SD, `auc` scale-free).

In [ ]:
display(pairs.round(4))

**Per CDI domain** (split means; descriptive).

In [ ]:
display(per_domain.pivot_table(index=["model", "comparison", "cdi_semantic", "n_categories"], columns="dataset",
                               values=["delta_within_half", "delta_cross_half"]).round(3))

## Figures

In [ ]:
def fig_within_vs_cross(name: str):
    """Dumbbell: within-half (hollow) vs cross-half (filled) Δ per dataset; one panel per model."""
    dss = COMPARISONS[name]
    fig, axes = plt.subplots(1, len(MODELS), figsize=(4.6 * len(MODELS), 0.45 * len(dss) + 1.7),
                             sharey=True, layout="constrained")
    for ax, m in zip(np.atleast_1d(axes), MODELS):
        s = summary[(summary.model == m) & (summary.comparison == name)].set_index("dataset")
        for y, ds in enumerate(dss):
            w, x = s.loc[ds, "delta_within_half"], s.loc[ds, "delta_cross_half"]
            ax.plot([w, x], [y, y], color="0.55", lw=2, zorder=1)
            ax.scatter([w], [y], s=70, facecolor="white", edgecolor="0.2", lw=1.5, zorder=3)
            ax.scatter([x], [y], s=70, facecolor=COLOR[ds], edgecolor="0.2", lw=1.0, zorder=4)
            ax.annotate(f"{s.loc[ds, 'cross_over_within']:.0%} kept", (max(w, x), y), xytext=(8, 0),
                        textcoords="offset points", va="center", fontsize=9, color="0.25")
        ax.set_yticks(range(len(dss)), [LABEL[d] for d in dss]); ax.set_ylim(len(dss) - 0.5, -0.5)  # first dataset on top (idempotent with sharey)
        ax.set_xlim(0, summary["delta_within_half"].max() * 1.35)
        ax.set_title(m.upper() if m == "clip" else "DINOv3", fontsize=11)
        ax.set_xlabel("Δ = between − within domain (cosine)")
        ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    handles = [plt.Line2D([], [], marker="o", ls="", mfc="white", mec="0.2", label="Within half (pairs can share scenes)"),
               plt.Line2D([], [], marker="o", ls="", mfc="0.6", mec="0.2", label="Cross half (no shared scenes)")]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, frameon=False, fontsize=9)
    n = summary[summary.comparison == name].n_categories.iloc[0]
    fig.suptitle(f"CDI-domain cluster strength with and without shared scenes ({n} categories, {K_SPLITS} splits)", fontsize=11)
    out = CTX_FIGURES / f"context_split_within_vs_cross_{name}.png"
    fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


def fig_auc(name: str):
    """AUC with 95% category-bootstrap CI per dataset; one panel per model."""
    dss = COMPARISONS[name]
    fig, axes = plt.subplots(1, len(MODELS), figsize=(4.6 * len(MODELS), 0.45 * len(dss) + 1.8),
                             sharey=True, sharex=True, layout="constrained")
    for ax, m in zip(np.atleast_1d(axes), MODELS):
        s = summary[(summary.model == m) & (summary.comparison == name)].set_index("dataset")
        for y, ds in enumerate(dss):
            ax.plot([s.loc[ds, "auc_ci_lo"], s.loc[ds, "auc_ci_hi"]], [y, y], color="0.35", lw=1.6, zorder=1)
            ax.scatter([s.loc[ds, "auc_cross_half"]], [y], s=80, facecolor=COLOR[ds], edgecolor="0.2", lw=1.0, zorder=3)
            ax.annotate(f"{s.loc[ds, 'auc_cross_half']:.3f}", (s.loc[ds, "auc_ci_hi"], y), xytext=(6, 0),
                        textcoords="offset points", va="center", fontsize=9, color="0.25")
        ax.set_yticks(range(len(dss)), [LABEL[d] for d in dss]); ax.set_ylim(len(dss) - 0.5, -0.5)  # first dataset on top (idempotent with sharey)
        ax.set_title(m.upper() if m == "clip" else "DINOv3", fontsize=11)
        ax.set_xlabel("AUC: P(between > within distance)")
        ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    fig.text(0.5, 0.0, "Intervals are per dataset. Dataset differences use a paired category bootstrap "
             "(context_split_dataset_differences.csv), which is tighter.", ha="center", va="bottom", fontsize=8.5, color="0.35")
    fig.get_layout_engine().set(rect=(0, 0.05, 1, 0.95))
    n = summary[summary.comparison == name].n_categories.iloc[0]
    fig.suptitle(f"Scale-free cluster strength, cross-half RDMs ({n} categories; 95% category-bootstrap CI)", fontsize=11)
    out = CTX_FIGURES / f"context_split_auc_{name}.png"
    fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


show_images([f(name) for name in COMPARISONS for f in (fig_within_vs_cross, fig_auc)], width=820)

## Supplementary figure (manuscript)

Subject split, 115 categories shared by BV, VEDB and THINGS. Top: cluster strength Δ within half vs cross half.
Bottom: scale-free AUC on the split-averaged cross-half RDM with 95% category-bootstrap CIs.

In [ ]:
def fig_supplement(name: str = "bv_vedb_things"):
    dss = COMPARISONS[name]; ylim = (len(dss) - 0.5, -0.5)
    fig, axes = plt.subplots(2, len(MODELS), figsize=(9.2, 5.6), sharey=True, layout="constrained")
    for col, m in enumerate(MODELS):
        s = summary[(summary.model == m) & (summary.comparison == name)].set_index("dataset")
        ax = axes[0, col]
        for y, ds in enumerate(dss):
            w, x = s.loc[ds, "delta_within_half"], s.loc[ds, "delta_cross_half"]
            ax.plot([w, x], [y, y], color="0.55", lw=2, zorder=1)
            ax.scatter([w], [y], s=60, facecolor="white", edgecolor="0.2", lw=1.4, zorder=3)
            ax.scatter([x], [y], s=60, facecolor=COLOR[ds], edgecolor="0.2", lw=1.0, zorder=4)
            ax.annotate(f"{s.loc[ds, 'cross_over_within']:.0%}", (max(w, x), y), xytext=(7, 0),
                        textcoords="offset points", va="center", fontsize=8.5, color="0.25")
        ax.set_xlim(0, summary.loc[summary.comparison == name, "delta_within_half"].max() * 1.3)
        ax.set_xlabel("Δ = between − within domain (cosine)", fontsize=9)
        ax = axes[1, col]
        for y, ds in enumerate(dss):
            ax.plot([s.loc[ds, "auc_ci_lo"], s.loc[ds, "auc_ci_hi"]], [y, y], color="0.35", lw=1.5, zorder=1)
            ax.scatter([s.loc[ds, "auc_cross_half"]], [y], s=65, facecolor=COLOR[ds], edgecolor="0.2", lw=1.0, zorder=3)
            ax.annotate(f"{s.loc[ds, 'auc_cross_half']:.3f}", (s.loc[ds, "auc_ci_hi"], y), xytext=(5, 0),
                        textcoords="offset points", va="center", fontsize=8.5, color="0.25")
        ax.set_xlim(0.62, 0.86)
        ax.set_xlabel("AUC: P(between > within distance)", fontsize=9)
        axes[0, col].set_title("CLIP" if m == "clip" else "DINOv3", fontsize=11)
    for k, ax in enumerate(axes.ravel()):
        ax.set_yticks(range(len(dss)), [LABEL[d] for d in dss]); ax.set_ylim(*ylim)
        ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
        ax.text(-0.02, 1.04, "ABCD"[k], transform=ax.transAxes, fontsize=12, fontweight="bold", ha="right", va="bottom")
    handles = [plt.Line2D([], [], marker="o", ls="", mfc="white", mec="0.2", label="Within half (pairs can share scenes)"),
               plt.Line2D([], [], marker="o", ls="", mfc="0.6", mec="0.2", label="Cross half (no shared scenes)")]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, frameon=False, fontsize=9)
    out = CTX_FIGURES / f"supp_context_split_{name}.png"
    fig.savefig(out, dpi=300); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


show_images([fig_supplement()], width=820)

### Review checklist

- Does the Stage 0 check pass (max |Δ| ~1e-8) for both models and both datasets?
- Is the BV split unit the subject ID, and is each split's category set fixed across splits?
- Are the two halves z-scored separately, and is the cross-half RDM symmetrized with its diagonal unused?
- Do bootstrap draws share category indices across datasets, with self-pairs dropped?
- Which conclusions change between raw Δ, Δ/SD and AUC?
- Caveat to carry into any write-up: the subject split removes *shared scenes*, not *typical settings*
  present in every home; VEDB sessions may share wearers or places, so its split may be less clean than BV's.